In [ ]:
import os
import sys
from pathlib import Path
from tokens import openai_key, HF_TOKEN

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

from revlm import *
tg_fvqa = TextGeneralizer(dataset_name="fvqa", openai_key=openai_key)
tg_aokvqa = TextGeneralizer(dataset_name="aokvqa", openai_key=openai_key)
PARQUET_DIR = Path("./data/related_text/parquet/")


process fvqa in 50 batches
process aokvqa in 100 batches


In [ ]:
# tg_fvqa.run_input()
# tg_fvqa.run_request()
fvqa_related_texts = tg_fvqa.get_related_texts()
# tg_fvqa.resubmit_request([14, 21, 22, 30, 38, 39, 47]) # resubmit failed / too-long requests

In [ ]:
# tg_aokvqa.run_input()
# tg_aokvqa.run_request()
aokvqa_related_texts = tg_aokvqa.get_related_texts()
# tg_aokvqa.resubmit_request([33, 42, 46])

In [ ]:
import pandas as pd

def related_texts_to_df(related_texts: dict) -> pd.DataFrame:
    rows = []
    for uid, variants in related_texts.items():
        rows.append(
            {
                "uid": uid,
                "variants": variants,
                "n_variants": len(variants),
            }
        )
    return pd.DataFrame(rows)

fvqa_df = related_texts_to_df(fvqa_related_texts)
aokvqa_df = related_texts_to_df(aokvqa_related_texts)

PARQUET_DIR.mkdir(parents=True, exist_ok=True)
fvqa_df.to_parquet(PARQUET_DIR / "fvqa.parquet", index=False)
aokvqa_df.to_parquet(PARQUET_DIR / "aokvqa.parquet", index=False)

In [ ]:
fvqa_df['variants'][0]

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi(token=HF_TOKEN)
repo_id = "JJoy333/RationaleVQA"
create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)


upload_folder(
    folder_path=str(PARQUET_DIR),
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="t_gen"
)
